# 05. Export YOLOv8 Model to TensorFlow Lite

This notebook converts the trained DhakaRoadNet YOLOv8 model into TensorFlow Lite files for a native Android app built with Kotlin and XML. The goal is to create Android-ready model files, verify them on real/sample images, and save a clean export report for your portfolio and higher-study documentation.

## Step 1: Understand the export goal

- `.pt` is the PyTorch/Ultralytics training checkpoint. It is good for training and Python testing.
- `.tflite` is the TensorFlow Lite model format. It is suitable for Android on-device inference.
- FP32 TFLite is the safest baseline.
- FP16 TFLite is smaller and usually a good first Android option.
- INT8 TFLite is the most Edge AI focused option, but it must be checked carefully because accuracy can drop.

In [1]:
# Run this cell with RUN_INSTALLS = True only if export imports fail.
# After installation, restart(ctrl + shift + P) the notebook kernel and run from the top again.

RUN_INSTALLS = False # Change to True if imports fail due to missing packages.

if RUN_INSTALLS:
    import subprocess
    import sys

    core_packages = [
        "ultralytics",
        "torch",
        "onnx",
        "onnxruntime",
    ]
    
    optional_packages = [
        "onnxslim",
        "onnx2tf",
        "tf_keras",
        "sng4onnx",
        "onnx_graphsurgeon",
        "ai-edge-litert",
        "pandas",
        "tabulate",
        "packaging",
        "opencv-python",
        "pillow",
        "matplotlib",
    ]

    print("Installing core packages first...")
    for package in core_packages:
        try:
            print(f"  Installing {package}...")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "--upgrade", package],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.STDOUT
            )
            print(f"    ✓ {package}")
        except subprocess.CalledProcessError:
            print(f"    ✗ {package} failed. Continuing...")

    print("\\nInstalling optional packages...")
    for package in optional_packages:
        try:
            print(f"  Installing {package}...")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "--upgrade", package],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.STDOUT
            )
            print(f"    ✓ {package}")
        except subprocess.CalledProcessError:
            print(f"    ✗ {package} skipped (optional)")

    print("\\nInstalling TensorFlow required by Ultralytics TFLite export...")
    if sys.version_info >= (3, 13):
        print(f"    ✗ Current Python is {sys.version.split()[0]}.")
        print("    TensorFlow TFLite export is much more reliable with Python 3.11 or 3.12.")
        print("    Create a Python 3.11/3.12 environment for this export notebook.")
    else:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "--upgrade", "tensorflow>=2.19.0", "protobuf<6"],
            )
            print("    ✓ tensorflow>=2.19.0")
        except subprocess.CalledProcessError:
            print("    ✗ TensorFlow installation failed. TFLite export will be skipped until TensorFlow is installed.")

    print("\\nInstallation complete!")
else:
    print("Install step skipped. Set RUN_INSTALLS = True if export dependencies are missing.")


Install step skipped. Set RUN_INSTALLS = True if export dependencies are missing.


## Step 2: Import libraries

These imports are used for exporting, file management, simple prediction checks, and writing a readable export report.

In [2]:
from datetime import datetime
from pathlib import Path
import json
import shutil
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

try:
    import tensorflow as tf
    TF_AVAILABLE = True
    TF_VERSION = tf.__version__
except Exception as tensorflow_error:
    tf = None
    TF_AVAILABLE = False
    TF_VERSION = None

print("Imports completed.")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("TensorFlow available:", TF_AVAILABLE)
print("TensorFlow version:", TF_VERSION)


Imports completed.
Torch: 2.12.0+cpu
CUDA available: False
TensorFlow available: True
TensorFlow version: 2.20.0


## Step 3: Configure paths and export settings

All important paths and settings are kept in one place. This makes the notebook easier to read and safer to modify later.

In [3]:
MODEL_RELATIVE_PATH = Path("model/checkpoints/yolov8n_dhakaroadnet_baseline/best.pt")
DATA_YAML_RELATIVE_PATH = Path("data/roboflow/data_yolov8.yaml")

IMAGE_SIZE = 640
BATCH_SIZE = 1
USE_NMS = True
EXPORT_DEVICE = "cpu"

EXPORT_DIR_RELATIVE_PATH = Path("model/exported/tflite")
REPORT_DIR_RELATIVE_PATH = Path("reports/tflite_export")

FP32_NAME = "dhakaroadnet_yolov8n_fp32.tflite"
FP16_NAME = "dhakaroadnet_yolov8n_fp16.tflite"
INT8_NAME = "dhakaroadnet_yolov8n_int8.tflite"

print("Export settings ready.")


Export settings ready.


## Step 4: Find the project root and check required files

This cell works whether Jupyter starts from the project root or from inside the `notebooks` folder.

In [4]:
def find_project_root(required_path: Path) -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / required_path).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {required_path}. Check the project folder.")


PROJECT_ROOT = find_project_root(MODEL_RELATIVE_PATH)
MODEL_PATH = PROJECT_ROOT / MODEL_RELATIVE_PATH
DATA_YAML_PATH = PROJECT_ROOT / DATA_YAML_RELATIVE_PATH
EXPORT_DIR = PROJECT_ROOT / EXPORT_DIR_RELATIVE_PATH
REPORT_DIR = PROJECT_ROOT / REPORT_DIR_RELATIVE_PATH
PREDICTION_DIR = REPORT_DIR / "predictions"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Model exists:", MODEL_PATH.exists(), MODEL_PATH)
print("Dataset YAML exists:", DATA_YAML_PATH.exists(), DATA_YAML_PATH)
print("Export directory:", EXPORT_DIR)
print("Report directory:", REPORT_DIR)


Project root: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI
Model exists: True C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\checkpoints\yolov8n_dhakaroadnet_baseline\best.pt
Dataset YAML exists: True C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\data\roboflow\data_yolov8.yaml
Export directory: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\exported\tflite
Report directory: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\reports\tflite_export


## Step 5: Check export dependencies

If any required package is missing, go back to the first code cell, set `RUN_INSTALLS = True`, run it, and restart the kernel.

In [5]:
import importlib.util
import sys
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
import sys

required_packages = {
    "ultralytics": None,
    "torch": None,
    "tensorflow": "2.19.0",
    "onnx": None,
    "onnxruntime": None,
    "onnxslim": None,
    "onnx2tf": None,
    "tf_keras": None,
}

dependency_rows = []
for package_name, minimum_version in required_packages.items():
    available = importlib.util.find_spec(package_name) is not None
    installed_version = None
    version_ok = available

    if available:
        try:
            installed_version = version(package_name)
        except PackageNotFoundError:
            installed_version = "importable"

    if available and minimum_version and installed_version not in (None, "importable"):
        version_ok = Version(installed_version) >= Version(minimum_version)

    dependency_rows.append(
        {
            "package": package_name,
            "available": available,
            "version": installed_version,
            "minimum_version": minimum_version,
            "ready": version_ok,
        }
    )

dependency_table = pd.DataFrame(dependency_rows)
python_version = sys.version.split()[0]
python_ok_for_tensorflow = sys.version_info < (3, 13)

print("Python version:", python_version)
if not python_ok_for_tensorflow:
    print("This Python version is too new for reliable TensorFlow/TFLite export.")
    print("Use Python 3.11 or 3.12 for this notebook, then install tensorflow>=2.19.0.")

if not bool(dependency_table.loc[dependency_table["package"] == "tensorflow", "ready"].iloc[0]):
    print("TensorFlow is missing or too old for TFLite export.")
    print("Set RUN_INSTALLS = True in the first code cell, run it, then restart the kernel.")

dependency_table


Python version: 3.12.10


,package,available,version,minimum_version,ready
0,ultralytics,True,8.4.64,NaN,True
1,torch,True,2.12.0,NaN,True
2,tensorflow,True,2.20.0,2.19.0,True
3,onnx,True,1.20.1,NaN,True
4,onnxruntime,True,1.24.3,NaN,True
5,onnxslim,True,0.1.94,NaN,True
6,onnx2tf,True,2.4.1,NaN,True
7,tf_keras,True,2.20.1,NaN,True


## Step 6: Load the trained YOLOv8 model

This is the same final `best.pt` model used in evaluation and Gradio testing.

In [6]:
model = YOLO(str(MODEL_PATH))
class_names = model.names

print("Model loaded:", MODEL_PATH.name)
print("Number of classes:", len(class_names))
print(class_names)


Model loaded: best.pt
Number of classes: 24
{0: 'Auto rickshaw', 1: 'Bicycle', 2: 'Bus', 3: 'Car', 4: 'Dog', 5: 'Garbage van', 6: 'Human', 7: 'Leguna', 8: 'Manhole', 9: 'Micro Bus', 10: 'Mini truck', 11: 'Minivan', 12: 'Motorbike', 13: 'Pickup truck', 14: 'Police car', 15: 'Pothole', 16: 'Rickshaw', 17: 'Road barrier', 18: 'SUV', 19: 'Speed Breaker', 20: 'Three wheelers -CNG-', 21: 'Truck', 22: 'Van', 23: 'Zebra Crossing'}


## Step 7: Save Android labels

The Android app needs the same class names as the model. This creates a simple `labels.txt` file with one class per line.

In [7]:
labels_path = EXPORT_DIR / "labels.txt"
ordered_labels = [class_names[index] for index in sorted(class_names)]
labels_path.write_text("\n".join(ordered_labels), encoding="utf-8")

print("Saved labels:", labels_path)
print("First labels:", ordered_labels[:5])


Saved labels: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\exported\tflite\labels.txt
First labels: ['Auto rickshaw', 'Bicycle', 'Bus', 'Car', 'Dog']


## Step 8: Create small helper functions

These helpers keep the export cells short. Each export will copy the generated `.tflite` file into a clean final filename.

In [8]:
from packaging.version import Version

export_records = []


def file_size_mb(path: Path) -> float:
    return round(path.stat().st_size / (1024 * 1024), 2)


def find_new_tflite_file(before_files, started_at):
    after_files = set(PROJECT_ROOT.rglob("*.tflite"))
    new_files = list(after_files - before_files)

    if not new_files:
        new_files = [p for p in after_files if p.stat().st_mtime >= started_at]

    if not new_files:
        raise FileNotFoundError("Export finished, but no new .tflite file was found.")

    return max(new_files, key=lambda p: p.stat().st_mtime)


def tensorflow_is_ready():
    if sys.version_info >= (3, 13):
        return False, f"Current Python is {sys.version.split()[0]}; use Python 3.11 or 3.12 for TensorFlow/TFLite export."
    if not TF_AVAILABLE or TF_VERSION is None:
        return False, "TensorFlow is not installed."
    if Version(TF_VERSION) < Version("2.19.0"):
        return False, f"TensorFlow {TF_VERSION} is installed, but Ultralytics requires tensorflow>=2.19.0."
    return True, f"TensorFlow {TF_VERSION} is ready."


def export_tflite_candidate(name, final_file_name, half=False, int8=False):
    print(f"Starting {name} export...")
    before_files = set(PROJECT_ROOT.rglob("*.tflite"))
    started_at = time.time()

    tf_ready, tf_message = tensorflow_is_ready()
    if not tf_ready:
        record = {
            "name": name,
            "status": "skipped",
            "path": "",
            "size_mb": None,
            "half": half,
            "int8": int8,
            "nms": USE_NMS,
            "imgsz": IMAGE_SIZE,
            "raw_export_result": "",
            "error": tf_message + " Set RUN_INSTALLS = True, run the first code cell, restart the kernel, then export again.",
        }
        print(f"Skipped {name}: {record['error']}")
        export_records.append(record)
        return record

    export_kwargs = {
        "format": "tflite",
        "imgsz": IMAGE_SIZE,
        "batch": BATCH_SIZE,
        "nms": USE_NMS,
        "device": EXPORT_DEVICE,
        "half": half,
        "int8": int8,
    }

    if int8:
        export_kwargs["data"] = str(DATA_YAML_PATH)

    try:
        raw_export_result = model.export(**export_kwargs)
        source_file = find_new_tflite_file(before_files, started_at)
        final_path = EXPORT_DIR / final_file_name
        shutil.copy2(source_file, final_path)

        record = {
            "name": name,
            "status": "success",
            "path": str(final_path),
            "size_mb": file_size_mb(final_path),
            "half": half,
            "int8": int8,
            "nms": USE_NMS,
            "imgsz": IMAGE_SIZE,
            "raw_export_result": str(raw_export_result),
            "error": "",
        }
        print(f"Finished {name}: {final_path} ({record['size_mb']} MB)")

    except Exception as exc:
        record = {
            "name": name,
            "status": "failed",
            "path": "",
            "size_mb": None,
            "half": half,
            "int8": int8,
            "nms": USE_NMS,
            "imgsz": IMAGE_SIZE,
            "raw_export_result": "",
            "error": str(exc),
        }
        print(f"{name} export failed:", exc)

    export_records.append(record)
    return record


## Step 9: Export FP32 TFLite

FP32 is the safest export. Use this first to confirm the conversion pipeline works.

In [9]:
fp32_record = export_tflite_candidate(
    name="FP32 TFLite",
    final_file_name=FP32_NAME,
    half=False,
    int8=False,
)
fp32_record


Starting FP32 TFLite export...
Ultralytics 8.4.64  Python-3.12.10 torch-2.12.0+cpu CPU (AMD Ryzen 3 3100 4-Core Processor)
Model summary (fused): 73 layers, 3,010,328 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\checkpoints\yolov8n_dhakaroadnet_baseline\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (6.0 MB)

ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  2.1s, saved as 'C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\checkpoints\yolov8n_dhakaroadnet_baseline\best.onnx' (11.8 MB)
requirements: Ultralytics requirement ['onnx2tf>=1.26.3,<1.29.0'] not found, attempting AutoUpdate...
  Attempting uninstall: onnx2tf
    Found existing installation: onnx2tf 2.4.1
    Uninstalling onnx2tf-2.4.1:
      Successfully uninstalled onnx2tf-2.4.1

requirements: AutoUpdate success  6.9s
WARNING requirements: Restart

{'name': 'FP32 TFLite',
 'status': 'success',
 'path': 'C:\\Users\\USER\\Desktop\\ML\\WorkSpace\\DhakaRoadNet-EdgeAI\\model\\exported\\tflite\\dhakaroadnet_yolov8n_fp32.tflite',
 'size_mb': 11.78,
 'half': False,
 'int8': False,
 'nms': True,
 'imgsz': 640,
 'raw_export_result': 'C:\\Users\\USER\\Desktop\\ML\\WorkSpace\\DhakaRoadNet-EdgeAI\\model\\checkpoints\\yolov8n_dhakaroadnet_baseline\\best_saved_model\\best_float32.tflite',
 'error': ''}

## Step 10: Export FP16 TFLite

FP16 is usually the best first Android candidate because it is smaller than FP32 and often keeps similar accuracy.

In [10]:
fp16_record = export_tflite_candidate(
    name="FP16 TFLite",
    final_file_name=FP16_NAME,
    half=True,
    int8=False,
)
fp16_record


Starting FP16 TFLite export...
Ultralytics 8.4.64  Python-3.12.10 torch-2.12.0+cpu CPU (AMD Ryzen 3 3100 4-Core Processor)
Model summary (fused): 73 layers, 3,010,328 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\checkpoints\yolov8n_dhakaroadnet_baseline\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (6.0 MB)

ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  1.9s, saved as 'C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\checkpoints\yolov8n_dhakaroadnet_baseline\best.onnx' (11.8 MB)

TensorFlow SavedModel: starting export with tensorflow 2.20.0...
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at 'C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\checkpoints\yolov8n_dhakaroadnet_baseline\best_saved_model'. The following endpoints are available:

* Endpoint 

{'name': 'FP16 TFLite',
 'status': 'success',
 'path': 'C:\\Users\\USER\\Desktop\\ML\\WorkSpace\\DhakaRoadNet-EdgeAI\\model\\exported\\tflite\\dhakaroadnet_yolov8n_fp16.tflite',
 'size_mb': 11.79,
 'half': True,
 'int8': False,
 'nms': True,
 'imgsz': 640,
 'raw_export_result': 'C:\\Users\\USER\\Desktop\\ML\\WorkSpace\\DhakaRoadNet-EdgeAI\\model\\checkpoints\\yolov8n_dhakaroadnet_baseline\\best_saved_model\\best_float16.tflite',
 'error': ''}

## Step 11: Export INT8 TFLite

INT8 is the strongest Edge AI option for size and speed. It uses your dataset YAML for calibration. If accuracy drops too much, use FP16 first in Android.

In [11]:
int8_record = export_tflite_candidate(
    name="INT8 TFLite",
    final_file_name=INT8_NAME,
    half=False,
    int8=True,
)
int8_record


Starting INT8 TFLite export...
Ultralytics 8.4.64  Python-3.12.10 torch-2.12.0+cpu CPU (AMD Ryzen 3 3100 4-Core Processor)
Model summary (fused): 73 layers, 3,010,328 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\checkpoints\yolov8n_dhakaroadnet_baseline\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (6.0 MB)
TensorFlow SavedModel: collecting INT8 calibration images from 'data=C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\data\roboflow\data_yolov8.yaml'
val: Fast image access  (ping: 0.10.0 ms, read: 101.019.7 MB/s, size: 65.8 KB)
val: Scanning C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\data\roboflow\valid\labels.cache... 195 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 195/195  0.0s
WARNING TensorFlow SavedModel: >300 images recommended for INT8 calibration, found 195 images.

ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onn

{'name': 'INT8 TFLite',
 'status': 'success',
 'path': 'C:\\Users\\USER\\Desktop\\ML\\WorkSpace\\DhakaRoadNet-EdgeAI\\model\\exported\\tflite\\dhakaroadnet_yolov8n_int8.tflite',
 'size_mb': 3.18,
 'half': False,
 'int8': True,
 'nms': True,
 'imgsz': 640,
 'raw_export_result': 'C:\\Users\\USER\\Desktop\\ML\\WorkSpace\\DhakaRoadNet-EdgeAI\\model\\checkpoints\\yolov8n_dhakaroadnet_baseline\\best_saved_model\\best_int8.tflite',
 'error': ''}

## Step 12: Save export summary

This summary makes the notebook useful for documentation, GitHub, and later Android decisions.

In [12]:
export_summary = pd.DataFrame(export_records)
summary_csv_path = REPORT_DIR / "export_summary.csv"
summary_json_path = REPORT_DIR / "export_summary.json"

export_summary.to_csv(summary_csv_path, index=False)
summary_json_path.write_text(json.dumps(export_records, indent=2), encoding="utf-8")

print("Saved CSV summary:", summary_csv_path)
print("Saved JSON summary:", summary_json_path)
export_summary


Saved CSV summary: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\reports\tflite_export\export_summary.csv
Saved JSON summary: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\reports\tflite_export\export_summary.json


,name,status,path,size_mb,half,int8,nms,imgsz,raw_export_result,error
0,FP32 TFLite,success,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,11.78,False,False,True,640,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
1,FP16 TFLite,success,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,11.79,True,False,True,640,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
2,INT8 TFLite,success,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,3.18,False,True,True,640,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,


## Step 13: Select sample images for prediction checks

A real export is not complete until the exported model is tested visually. This cell selects a few test images from your Roboflow test split.

In [13]:
TEST_IMAGE_DIR = PROJECT_ROOT / "data" / "roboflow" / "test" / "images"
sample_images = sorted(TEST_IMAGE_DIR.glob("*.jpg"))[:5]

print("Sample image directory:", TEST_IMAGE_DIR)
print("Number of selected sample images:", len(sample_images))
for path in sample_images:
    print(path.name)


Sample image directory: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\data\roboflow\test\images
Number of selected sample images: 5
13_jpg.rf.a83769ea84ab33ba8b24306f38f8a107.jpg
15_jpg.rf.10babafabc168b49e3eec7b7e9459d4e.jpg
1_jpg.rf.35574b68fbec2a2d4e24066ad817cfbb.jpg
20_jpg.rf.bd8db1759b0cd63298d9c75d1253b4dc.jpg
27_jpg.rf.4cca0863ca9530774f72122824824938.jpg


## Step 14: Prediction helper for `.pt` and `.tflite`

This helper saves annotated prediction images. If a TFLite model cannot be loaded by Ultralytics on your machine, the error is saved in the report instead of stopping the notebook.

In [14]:
prediction_records = []


def run_prediction_check(model_path, model_label, images, confidence=0.25):
    print(f"Checking predictions for {model_label}...")
    try:
        check_model = YOLO(str(model_path))
        for image_path in images:
            started_at = time.perf_counter()
            results = check_model.predict(
                source=str(image_path),
                imgsz=IMAGE_SIZE,
                conf=confidence,
                device="cpu",
                verbose=False,
            )
            elapsed_seconds = time.perf_counter() - started_at
            result = results[0]
            annotated_bgr = result.plot()
            annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

            output_name = f"{model_label}_{image_path.stem}.jpg".replace(" ", "_").lower()
            output_path = PREDICTION_DIR / output_name
            Image.fromarray(annotated_rgb).save(output_path)

            detection_count = 0 if result.boxes is None else len(result.boxes)
            prediction_records.append(
                {
                    "model_label": model_label,
                    "model_path": str(model_path),
                    "image": image_path.name,
                    "status": "success",
                    "detections": int(detection_count),
                    "time_seconds": round(float(elapsed_seconds), 4),
                    "output_path": str(output_path),
                    "error": "",
                }
            )
        print(f"Finished prediction check for {model_label}.")

    except Exception as exc:
        prediction_records.append(
            {
                "model_label": model_label,
                "model_path": str(model_path),
                "image": "",
                "status": "failed",
                "detections": None,
                "time_seconds": None,
                "output_path": "",
                "error": str(exc),
            }
        )
        print(f"Prediction check failed for {model_label}:", exc)


## Step 15: Run visual prediction checks

This compares the original PyTorch checkpoint with each successful TFLite export.

In [15]:
if sample_images:
    run_prediction_check(MODEL_PATH, "pt_best", sample_images)

    for record in export_records:
        if record["status"] == "success" and record["path"]:
            model_label = record["name"].replace(" TFLite", "").lower()
            run_prediction_check(Path(record["path"]), model_label, sample_images)
else:
    print("No sample images found. Add test images before running prediction checks.")

prediction_summary = pd.DataFrame(prediction_records)
prediction_summary_path = REPORT_DIR / "prediction_check_summary.csv"
prediction_summary.to_csv(prediction_summary_path, index=False)

print("Saved prediction summary:", prediction_summary_path)
prediction_summary


Checking predictions for pt_best...
Finished prediction check for pt_best.
Checking predictions for fp32...
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\exported\tflite\dhakaroadnet_yolov8n_fp32.tflite for TensorFlow Lite inference...
Finished prediction check for fp32.
Checking predictions for fp16...
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\model\exported\tflite\dhakaroadnet_yolov8n_fp16.tflite for TensorFlow Lite inference...
Finished prediction check for fp16.
Checking predictions for int8...
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicit

,model_label,model_path,image,status,detections,time_seconds,output_path,error
0,pt_best,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,13_jpg.rf.a83769ea84ab33ba8b24306f38f8a107.jpg,success,2,0.1439,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
1,pt_best,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,15_jpg.rf.10babafabc168b49e3eec7b7e9459d4e.jpg,success,0,0.0954,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
2,pt_best,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,1_jpg.rf.35574b68fbec2a2d4e24066ad817cfbb.jpg,success,2,0.0962,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
3,pt_best,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,20_jpg.rf.bd8db1759b0cd63298d9c75d1253b4dc.jpg,success,1,0.0897,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
4,pt_best,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,27_jpg.rf.4cca0863ca9530774f72122824824938.jpg,success,3,0.1952,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
5,fp32,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,13_jpg.rf.a83769ea84ab33ba8b24306f38f8a107.jpg,success,2,0.2339,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
6,fp32,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,15_jpg.rf.10babafabc168b49e3eec7b7e9459d4e.jpg,success,0,0.1412,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
7,fp32,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,1_jpg.rf.35574b68fbec2a2d4e24066ad817cfbb.jpg,success,2,0.1474,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
8,fp32,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,20_jpg.rf.bd8db1759b0cd63298d9c75d1253b4dc.jpg,success,1,0.1397,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,
9,fp32,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,27_jpg.rf.4cca0863ca9530774f72122824824938.jpg,success,3,0.1391,C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNe...,


## Step 16: Write final export report

This markdown report explains what was exported and gives a practical recommendation for Android development.

In [ ]:
successful_exports = [record for record in export_records if record["status"] == "success"]

if any(record["name"] == "FP16 TFLite" and record["status"] == "success" for record in export_records):
    recommendation = "Use FP16 TFLite first in the Android app. Keep FP32 for debugging and test INT8 carefully."
elif any(record["name"] == "FP32 TFLite" and record["status"] == "success" for record in export_records):
    recommendation = "Use FP32 TFLite for initial Android integration, then retry FP16/INT8 export."
else:
    recommendation = "No TFLite export succeeded. Fix dependency/export errors before Android integration."

report_path = REPORT_DIR / "export_summary.md"
report_text = f"""# DhakaRoadNet TFLite Export Summary

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Source Model

- Model: `{MODEL_PATH}`
- Dataset YAML: `{DATA_YAML_PATH}`
- Image size: `{IMAGE_SIZE}`
- NMS included in export: `{USE_NMS}`

## Exported Files

{export_summary.to_markdown(index=False) if not export_summary.empty else 'No export records found.'}

## Android Files

- TFLite models: `{EXPORT_DIR}`
- Labels file: `{labels_path}`
"""

report_path.write_text(report_text, encoding="utf-8")
print("Saved markdown report:", report_path)
print(recommendation)


Saved markdown report: C:\Users\USER\Desktop\ML\WorkSpace\DhakaRoadNet-EdgeAI\reports\tflite_export\export_summary.md
Use FP16 TFLite first in the Android app. Keep FP32 for debugging and test INT8 carefully.
